To visualize changes in distance and speed over time during fish fights

In [20]:
import pandas as pd
import numpy as np
import cv2
import subprocess
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle
%matplotlib widget
import os
from matplotlib.widgets import Slider, Button
import matplotlib.animation as animation
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib.patches as patches
import multiprocessing
from concurrent.futures import ThreadPoolExecutor, as_completed
import tqdm
from tqdm import tqdm
import re
import threading




Data processing

In [21]:
csv_path = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_30dpf_AB_pattern_vs_nopattern/session_trimmed_output/trajectories/trajectories_csv/trajectories.csv"
input_video_path = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_150mm_30dpf_20250516/Analysis/output_trimmed.mp4"
output_video_path = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_150mm_30dpf_20250516/Analysis/output_annotated.mp4"
fps = 60

data = pd.read_csv(csv_path)
fish1 = data.iloc[:, [1, 2]].values
fish2 = data.iloc[:, [3, 4]].values

fish1_corrected = np.nan_to_num(fish1, nan=0.0)
fish2_corrected = np.nan_to_num(fish2, nan=0.0)
distance = np.linalg.norm(fish1 - fish2, axis=1)
speed1 = np.linalg.norm(np.diff(fish1, axis=0, prepend=fish1[0:1]), axis=1)
speed2 = np.linalg.norm(np.diff(fish2, axis=0, prepend=fish2[0:1]), axis=1)

# Replace NaNs with zeros or interpolate (if applicable)
distance = np.nan_to_num(distance)
speed1 = np.nan_to_num(speed1)
speed2 = np.nan_to_num(speed2)
print(len(distance))


FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_30dpf_AB_pattern_vs_nopattern/session_trimmed_output/trajectories/trajectories_csv/trajectories.csv'

Converting:   2%|▏         | 11851/649553 [01:46<1:35:09, 111.70frame/s]

FFmpeg: frame=11853
FFmpeg: fps=111.87
FFmpeg: stream_0_0_q=-1.0
FFmpeg: bitrate=5028.0kbits/s
FFmpeg: total_size=124138456
FFmpeg: out_time_us=197516667
FFmpeg: out_time_ms=197516667
FFmpeg: out_time=00:03:17.516667
FFmpeg: dup_frames=0
FFmpeg: drop_frames=0
FFmpeg: speed=1.86x
FFmpeg: progress=end


Intermediate checks

In [ ]:
# cap = cv2.VideoCapture(output_video_path)
# i = 0
# while True:
#     ret, _ = cap.read()
#     if not ret:
#         break
#     i += 1
# print(f"Actual readable frames: {i}")


In [ ]:
# cap = cv2.VideoCapture(output_video_path)
# fps = cap.get(cv2.CAP_PROP_FPS)
# frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
# num_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# min_len = min(num_frames, len(distance), len(speed1), len(speed2))
# distance = distance[:min_len]
# speed1 = speed1[:min_len]
# speed2 = speed2[:min_len]

# for i in range(min_len):
#     ret, frame = cap.read()
#     # if not ret:
#         print(f"Stopped early at frame {i}")
#         break
# print(min_len)
# print(fps)

In [ ]:
print(len(distance))  # Should be 18000
print(np.unique(distance[-100:]))  # If only one value: padding happened


Fight detection logic and annotation

In [ ]:
def detect_fights(distance, speed1, speed2, window=20, min_bout_len=1):
    
    fight_frames = []
    for i in range(window, len(distance)):
        d_diff = distance[i - window] - distance[i]
        s1_now = speed1[i]
        s2_now = speed2[i]
        s1_diff = s1_now - speed1[i - window]
        s2_diff = s2_now - speed2[i - window]

        if (
            d_diff > 5 and
            (s1_diff > 5 or s2_diff > 5)
    ):
            fight_frames.append(i)

    bouts = []
    if not fight_frames:
        return bouts

    start = fight_frames[0]
    prev = fight_frames[0]
    for f in fight_frames[1:]:
        if f - prev > 1:
            if prev - start + 1 >= min_bout_len:
                bouts.append((start, prev))
            start = f
        prev = f
    if prev - start + 1 >= min_bout_len:
        bouts.append((start, prev))

    return bouts



# --- Detect fights on actual data ---
fight_bouts = detect_fights(distance, speed1, speed2)
print(f"Detected {len(fight_bouts)} fight bouts")


In [ ]:
#generates csv with features for learning model

csv_path = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_30dpf_AB_pattern_vs_nopattern/idtracker/session_cropped_fight_annotated_2/trajectories/trajectories_csv/trajectories.csv"
fps = 60

data = pd.read_csv(csv_path)
fish1 = data.iloc[:, [1, 2]].values
fish2 = data.iloc[:, [3, 4]].values

fish1 = np.nan_to_num(fish1, nan=0.0)
fish2 = np.nan_to_num(fish2, nan=0.0)

# ---- FEATURES ----

# distance between fish
distance = np.linalg.norm(fish1 - fish2, axis=1)

# speed (euclidean distance per frame)
speed1 = np.linalg.norm(np.diff(fish1, axis=0, prepend=fish1[0:1]), axis=1) * fps
speed2 = np.linalg.norm(np.diff(fish2, axis=0, prepend=fish2[0:1]), axis=1) * fps

# acceleration
acc1 = np.diff(speed1, prepend=speed1[0]) * fps
acc2 = np.diff(speed2, prepend=speed2[0]) * fps

# heading
dx1 = np.diff(fish1[:,0], prepend=fish1[0,0])
dy1 = np.diff(fish1[:,1], prepend=fish1[0,1])
heading1 = np.arctan2(dy1, dx1)

dx2 = np.diff(fish2[:,0], prepend=fish2[0,0])
dy2 = np.diff(fish2[:,1], prepend=fish2[0,1])
heading2 = np.arctan2(dy2, dx2)

# relative heading
rel_heading_1tow2 = (heading2 - heading1 + np.pi) % (2*np.pi) - np.pi
rel_heading_2tow1 = (heading1 - heading2 + np.pi) % (2*np.pi) - np.pi

# ---- FIGHT LABELS ----

# simple detection using your rule-based function
def detect_fights(distance, speed1, speed2, window=20, min_bout_len=1):
    fight_frames = []
    for i in range(window, len(distance)):
        d_diff = distance[i - window] - distance[i]
        s1_now = speed1[i]
        s2_now = speed2[i]
        s1_diff = s1_now - speed1[i - window]
        s2_diff = s2_now - speed2[i - window]
        if (
            d_diff > 5 and
            (s1_diff > 5 or s2_diff > 5)
        ):
            fight_frames.append(i)
    bouts = []
    if not fight_frames:
        return bouts
    start = fight_frames[0]
    prev = fight_frames[0]
    for f in fight_frames[1:]:
        if f - prev > 1:
            if prev - start + 1 >= min_bout_len:
                bouts.append((start, prev))
            start = f
        prev = f
    if prev - start + 1 >= min_bout_len:
        bouts.append((start, prev))
    return bouts

fight_bouts = detect_fights(distance, speed1, speed2)

# frame-level fight labels
fight_labels = np.zeros(len(distance), dtype=int)
for start, end in fight_bouts:
    fight_labels[start:end+1] = 1

# ---- FINAL DATAFRAME ----

df = pd.DataFrame({
    'frame': np.arange(len(distance)),
    'x1': fish1[:,0],
    'y1': fish1[:,1],
    'x2': fish2[:,0],
    'y2': fish2[:,1],
    'inter_animal_distance': distance,
    'speed1': speed1,
    'speed2': speed2,
    'acc1': acc1,
    'acc2': acc2,
    'heading1': heading1,
    'heading2': heading2,
    'relative_heading_1tow2': rel_heading_1tow2,
    'relative_heading_2tow1': rel_heading_2tow1,
    'fight_label': fight_labels
})

# Save to a new path of your choice
output_csv_path = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_30dpf_AB_pattern_vs_nopattern/analysis/features_with_labels_2.csv"
df.to_csv(output_csv_path, index=False)
print("Saved features_with_labels.csv successfully!")


In [ ]:
import cv2

# Paths
input_path = input_video_path   
output_path = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_150mm_30dpf_pattern_vs_no_pattern_20250612/output_annotated.mp4"


# Open video
cap = cv2.VideoCapture(input_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

def is_fight_frame(i):
    return any(start <= i <= end for start, end in fight_bouts)

frame_idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    if is_fight_frame(frame_idx):
        cv2.putText(frame, 'FIGHT', (20, 45), cv2.FONT_HERSHEY_SIMPLEX,
                    1.5, (0, 0, 255), 3, cv2.LINE_AA)

    out.write(frame)
    frame_idx += 1

cap.release()
out.release()
print(f"Annotated video saved as: {output_path}")

In [ ]:
import cv2
import multiprocessing

# CONFIG
input_path = input_video_path
output_base = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_150mm_30dpf_pattern_vs_no_pattern_20250612"

# Example fight_bouts definition for testing
fight_bouts = [(100, 200), (1000, 1200)]  # replace with your actual bouts

def is_fight_frame(i):
    return any(start <= i <= end for start, end in fight_bouts)

def process_chunk(start_frame, end_frame, chunk_idx):
    cap = cv2.VideoCapture(input_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    
    chunk_output = f"{output_base}/chunk_{chunk_idx}.mp4"
    out = cv2.VideoWriter(chunk_output, fourcc, fps, (width, height))
    
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    frame_idx = start_frame
    
    while frame_idx <= end_frame:
        ret, frame = cap.read()
        if not ret:
            break
        
        if is_fight_frame(frame_idx):
            cv2.putText(frame, 'FIGHT', (20, 45), cv2.FONT_HERSHEY_SIMPLEX,
                        1.5, (0, 0, 255), 3, cv2.LINE_AA)
        
        out.write(frame)
        frame_idx += 1
    
    cap.release()
    out.release()
    print(f"Chunk {chunk_idx} done: {chunk_output}")

if __name__ == "__main__":
    cap = cv2.VideoCapture(input_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    
    num_chunks = multiprocessing.cpu_count()
    chunk_size = total_frames // num_chunks
    
    jobs = []
    for i in range(num_chunks):
        start = i * chunk_size
        end = (i + 1) * chunk_size - 1 if i != num_chunks - 1 else total_frames - 1
        p = multiprocessing.Process(target=process_chunk, args=(start, end, i))
        p.start()
        jobs.append(p)
    
    for p in jobs:
        p.join()
    
    print("All chunks processed. Now concatenate with ffmpeg if needed.")


In [ ]:
# Constants
FPS = 60
DURATION_MINUTES = 5
TOTAL_FRAMES = DURATION_MINUTES * 60 * FPS
WINDOW_SIZE = 60  # Number of frames visible at a time


# Setup video writer
plot_video_path = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_150mm_30dpf_20250516/fight_dynamic_plot.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
plot_width = 1280
plot_height = 480
plot_writer = cv2.VideoWriter(plot_video_path, fourcc, FPS, (plot_width, plot_height))

# Setup plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(plot_width/100, plot_height/100), dpi=100)
canvas = FigureCanvas(fig)

# Pre-draw fight spans (they’ll only appear when in view)
for start, end in fight_bouts:
    ax1.axvspan(start, end, color='red', alpha=0.1)
    ax2.axvspan(start, end, color='red', alpha=0.1)

# Plot setup
ax1.set_ylabel("Distance (px)")
ax2.set_ylabel("Speed (px/frame)")
ax2.set_xlabel("Frame")
ax1.set_ylim(0, np.max(distance) * 1.05)  # 5% padding
ax2.set_ylim(0, max(np.max(speed1), np.max(speed2)) * 1.05)


line_dist, = ax1.plot([], [], color='red', label='Distance')
line_s1, = ax2.plot([], [], color='blue', label='Speed 1')
line_s2, = ax2.plot([], [], color='green', label='Speed 2')
vline1 = ax1.axvline(0, color='black', linestyle='--')
vline2 = ax2.axvline(0, color='black', linestyle='--')
ax1.legend()
ax2.legend()

# Frame-by-frame plot generation with sliding window
for i in range(1, TOTAL_FRAMES):
    start_idx = max(0, i - WINDOW_SIZE)
    xs = np.arange(start_idx, i)

    line_dist.set_data(xs, distance[start_idx:i])
    line_s1.set_data(xs, speed1[start_idx:i])
    line_s2.set_data(xs, speed2[start_idx:i])
    vline1.set_xdata([i])
    vline2.set_xdata([i])

    ax1.set_xlim(start_idx, start_idx + WINDOW_SIZE)
    ax2.set_xlim(start_idx, start_idx + WINDOW_SIZE)

    canvas.draw()
    buf = np.frombuffer(canvas.tostring_rgb(), dtype=np.uint8)
    frame = buf.reshape(canvas.get_width_height()[::-1] + (3,))
    frame = cv2.resize(frame, (plot_width, plot_height))
    plot_writer.write(frame)

# Finalize
plot_writer.release()
plt.close()
print(f"Video saved to {plot_video_path}")


Trajectory animations

In [ ]:
import cv2
import numpy as np

def draw_trails(video_path, trajectories, fight_frames, output_path, trail_length=60):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    # Base colors
    colors = [(255, 0, 0), (0, 255, 0)]  # Fish 1: Blue, Fish 2: Green
    red = (0, 0, 255)

    # To store past positions
    trails = [[], []]  # For 2 fish

    if len(fight_frames) < total_frames:
        print(f"Warning: fight_frames has only {len(fight_frames)} entries, video has {total_frames} frames.")

    frame_idx = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_idx >= len(trajectories[0]):
            break

        for i in range(2):  # For each fish
            pos = trajectories[i][frame_idx]
            trails[i].append(pos)

            if len(trails[i]) > trail_length:
                trails[i].pop(0)

            for j in range(1, len(trails[i])):
                pt1_raw = trails[i][j - 1]
                pt2_raw = trails[i][j]

                # Check validity
                if (
                    np.all(np.isfinite(pt1_raw)) and np.all(np.isfinite(pt2_raw)) and
                    0 <= pt1_raw[0] < w and 0 <= pt1_raw[1] < h and
                    0 <= pt2_raw[0] < w and 0 <= pt2_raw[1] < h
                ):
                    pt1 = tuple(np.int32(pt1_raw))
                    pt2 = tuple(np.int32(pt2_raw))

                    # Fade factor
                    fade = 1 - (len(trails[i]) - j) / trail_length
                    fade = max(fade, 0.2)

                    # Color selection
                    in_fight = frame_idx < len(fight_frames) and fight_frames[frame_idx]
                    base_color = red if in_fight else colors[i]
                    color = tuple(int(c * fade) for c in base_color)

                    cv2.line(frame, pt1, pt2, color, 2)


        
        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    print(f"Trails video saved to {output_path}")

video_path = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_150mm_30dpf_20250516/Analysis/output_trimmed.mp4"
output_path = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_150mm_30dpf_20250516/Analysis/fight_annotated_full_video.mp4"
trajectories = [fish1_corrected, fish2_corrected]

n_frames= len(fish1_corrected)
fight_frames = np.zeros(n_frames, dtype=bool)

for start, end in fight_bouts:
    fight_frames[start:end+1] = True

draw_trails(video_path, trajectories, fight_frames, output_path, trail_length=60)


Crop region of interest

In [ ]:
input_crop_path = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_150mm_30dpf_20250516/raw_data/out_id0_60fps_20250516113903.avi"
output_video_path = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_150mm_30dpf_20250516/raw_data/cropped_dish1.mp4"
cap = cv2.VideoCapture(input_crop_path)
ret, frame = cap.read()
fps = 60
# Let user draw ROI on the first frame
roi = cv2.selectROI("Select Dish", frame, fromCenter=False, showCrosshair=True)
cv2.destroyAllWindows()
cv2.waitKey(1)
x, y, w, h = roi

# Now define crop function
def crop_frame(frame):
    return frame[y:y+h, x:x+w]
    
# Create and save cropped video using crop_frame function
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video_path, fourcc, fps, (w, h))

while True:
    ret, frame = cap.read()
    if not ret:
        break
    cropped = crop_frame(frame)
    out.write(cropped)
out.release()
cap.release()
cap = cv2.VideoCapture(output_video_path)
# Save ROI crop values
with open("crop_info.txt", "w") as f:
    f.write(f"{x},{y},{w},{h}")

Parallel processing for video cropping

In [ ]:
input_video = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_150mm_30dpf_20250516/raw_data/out_id0_60fps_20250516113903.avi"
output_dir = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_150mm_30dpf_20250516/raw_data"
ffmpeg_path = "ffmpeg"  # Ensure ffmpeg is installed and in PATH

# === Step 1: Read first frame for ROI selection ===
cap = cv2.VideoCapture(input_video)
ret, frame = cap.read()
cap.release()

if not ret or frame is None:
    raise ValueError("Could not read the first frame from the video.")

# === Step 2: Let user select ROIs ===
dish_rois = []
num_dishes = 3  # Or 6

for i in range(num_dishes):
    roi = cv2.selectROI(f"Select ROI for Dish {i+1}", frame, fromCenter=False, showCrosshair=True)
    cv2.destroyAllWindows()
    cv2.waitKey(1)
    if sum(roi) == 0:
        raise ValueError(f"Invalid ROI selected for dish {i+1}")
    dish_rois.append((i + 1, roi))  # (dish_id, (x, y, w, h))

# === Step 3: Save ROI info ===
with open("dish_rois.txt", "w") as f:
    for dish_id, (x, y, w, h) in dish_rois:
        f.write(f"{dish_id},{x},{y},{w},{h}\n")

# === Step 4: Define cropping function ===
def crop_dish(dish_id, roi):
    x, y, w, h = map(int, roi)
    output_path = os.path.join(output_dir, f"dish{dish_id}.mp4")
    
    cmd = [
        ffmpeg_path, "-hide_banner", "-y",
        "-i", input_video,
        "-vf", f"crop={w}:{h}:{x}:{y}",
        "-c:v", "libx264",         #  More reliable with .avi input
        "-crf", "18",
        "-preset", "fast",
        "-c:a", "copy",
        output_path
    ]
    
    print(f"Cropping dish {dish_id}...")
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    if result.returncode != 0:
        print(f"FFmpeg failed for dish {dish_id}:\n{result.stdout}")
    else:
        print(f"Dish {dish_id} cropped to {output_path}")

# === Step 5: Run crops in parallel ===
os.makedirs(output_dir, exist_ok=True)

with ThreadPoolExecutor(max_workers=min(num_dishes, 6)) as executor:
    futures = [executor.submit(crop_dish, dish_id, roi) for dish_id, roi in dish_rois]

In [ ]:
input_video = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_150mm_30dpf_20250516/raw_data/out_id0_60fps_20250516113903.avi"
output_dir = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_150mm_30dpf_20250516/raw_data"
ffmpeg_path = "ffmpeg"  # Ensure ffmpeg is installed and in PATH

# === Step 1: Convert .avi to .mp4 if needed ===
base_name = os.path.splitext(os.path.basename(input_video))[0]
converted_mp4 = os.path.join(os.path.dirname(input_video), f"{base_name}_converted.mp4")

if input_video.endswith(".avi"):
    print("Converting .avi to .mp4...")
    # Get frame count
    cap = cv2.VideoCapture(input_video)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap = cv2.VideoCapture(input_video)
    fps = cap.get(cv2.CAP_PROP_FPS)
    cap.release()

    convert_cmd = [
        ffmpeg_path,
        "-hwaccel", "videotoolbox",             # enable Apple hardware acceleration
        "-i", input_video,
        "-c:v", "h264_videotoolbox",            # use hardware encoder
        "-b:v", "15000k",                        # higher bitrate for quality
        "-c:a", "copy",
        "-y", converted_mp4,
        "-progress", "pipe:1", "-nostats"
  ]


    progress = tqdm(
        total=total_frames,
        desc="Converting",
        unit="frame",
        dynamic_ncols=True,
        miniters=1,
        smoothing=0.05,
    )

    frame_regex = re.compile(r"frame=\s*(\d+)")

    def update_conversion(pipe):
        for line in iter(pipe.readline, b''):
            try:
                line = line.decode('utf-8').strip()
                #print("FFmpeg:", line)  #DEBUG
                if line.startswith("out_time_ms"):
                    microseconds = int(line.split("=")[1])
                    frame_est = int((microseconds / 1_000_000) * fps)
                    progress.update(max(0, frame_est - progress.n))  
                    progress.refresh()
            except:
               continue
        pipe.close()

    proc = subprocess.Popen(
       convert_cmd,
       stdout=subprocess.PIPE,
       stderr=subprocess.DEVNULL
    )

    thread = threading.Thread(target=update_conversion, args=(proc.stdout,))
    thread.start()
    proc.wait()
    thread.join()
    progress.close()

    if proc.returncode != 0:
        raise RuntimeError("FFmpeg conversion failed.")

    print(f"Conversion complete: {converted_mp4}")
    input_video = converted_mp4

# === Step 2: Select ROIs ===
cap = cv2.VideoCapture(input_video)
ret, frame = cap.read()
cap.release()

if not ret or frame is None:
    raise ValueError("Could not read first frame.")

num_dishes = 3
dish_rois = []

for i in range(num_dishes):
    roi = cv2.selectROI(f"Select ROI for Dish {i+1}", frame, fromCenter=False, showCrosshair=True)
    cv2.destroyAllWindows()
    cv2.waitKey(1)
    if sum(roi) == 0:
        raise ValueError(f"Invalid ROI for dish {i+1}")
    dish_rois.append((i + 1, roi))

with open("dish_rois.txt", "w") as f:
    for dish_id, (x, y, w, h) in dish_rois:
        f.write(f"{dish_id},{x},{y},{w},{h}\n")

# === Step 3: Cropping with tqdm progress per job ===
def crop_dish(dish_id, roi):
    x, y, w, h = map(int, roi)
    output_path = os.path.join(output_dir, f"dish{dish_id}.mp4")
    cmd = [
        ffmpeg_path, "-i", input_video,
        "-vf", f"crop={w}:{h}:{x}:{y}",
        "-c:v", "h264_videotoolbox",
        "-b:v", "20M",
        "-c:a", "copy",
        "-y", output_path
    ]
    # Run FFmpeg and show progress (frame-based estimate)
    cap = cv2.VideoCapture(input_video)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    
    progress = tqdm(total=total_frames, desc=f"Cropping Dish {dish_id}", unit="frame", position=dish_id, leave=True)
    frame_regex = re.compile(r"frame=\s*(\d+)")
    
    def update(pipe):
        for line in iter(pipe.readline, b''):
            try:
                line = line.decode('utf-8')
                match = frame_regex.search(line)
                if match:
                    frame = int(match.group(1))
                    progress.n = frame
                    progress.refresh()
            except:
                continue
        pipe.close()

    proc = subprocess.Popen(cmd, stderr=subprocess.PIPE, stdout=subprocess.DEVNULL)
    thread = threading.Thread(target=update, args=(proc.stderr,))
    thread.start()
    proc.wait()
    thread.join()
    progress.close()

    if proc.returncode != 0:
        print(f"FFmpeg failed for dish {dish_id}")
    else:
        print(f"Dish {dish_id} saved to {output_path}")

# === Step 4: Run all crops in parallel ===
os.makedirs(output_dir, exist_ok=True)
with ThreadPoolExecutor(max_workers=min(num_dishes, 6)) as executor:
    futures = [executor.submit(crop_dish, dish_id, roi) for dish_id, roi in dish_rois]

In [ ]:
input_video = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_150mm_30dpf_20250516/raw_data/out_id0_60fps_20250516113903_converted.mp4"
output_dir = "/Volumes/jlarsch/default/D2c/Deeksha/Territory_assay/TA_200mm_150mm_30dpf_20250516/raw_data"
ffmpeg_path = "ffmpeg"  # Ensure ffmpeg is installed and in PATH

# === Step 2: Select ROIs ===
cap = cv2.VideoCapture(input_video)
ret, frame = cap.read()
cap.release()

if not ret or frame is None:
    raise ValueError("Could not read first frame.")

num_dishes = 3
dish_rois = []

for i in range(num_dishes):
    roi = cv2.selectROI(f"Select ROI for Dish {i+1}", frame, fromCenter=False, showCrosshair=True)
    cv2.destroyAllWindows()
    cv2.waitKey(1)
    if sum(roi) == 0:
        raise ValueError(f"Invalid ROI for dish {i+1}")
    dish_rois.append((i + 1, roi))

with open("dish_rois.txt", "w") as f:
    for dish_id, (x, y, w, h) in dish_rois:
        f.write(f"{dish_id},{x},{y},{w},{h}\n")

# === Step 3: Cropping with tqdm progress per job ===
def crop_dish(dish_id, roi):
    x, y, w, h = map(int, roi)
    output_path = os.path.join(output_dir, f"dish{dish_id}.mp4")
    cmd = [
        ffmpeg_path, "-i", input_video,
        "-vf", f"crop={w}:{h}:{x}:{y}",
        "-c:v", "h264_videotoolbox",
        "-b:v", "20M",
        "-c:a", "copy",
        "-y", output_path
    ]
    # Run FFmpeg and show progress (frame-based estimate)
    cap = cv2.VideoCapture(input_video)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    
    progress = tqdm(total=total_frames, desc=f"Cropping Dish {dish_id}", unit="frame", position=dish_id, leave=True)
    frame_regex = re.compile(r"frame=\s*(\d+)")
    
    def update(pipe):
        for line in iter(pipe.readline, b''):
            try:
                line = line.decode('utf-8')
                match = frame_regex.search(line)
                if match:
                    frame = int(match.group(1))
                    progress.n = frame
                    progress.refresh()
            except:
                continue
        pipe.close()

    proc = subprocess.Popen(cmd, stderr=subprocess.PIPE, stdout=subprocess.DEVNULL)
    thread = threading.Thread(target=update, args=(proc.stderr,))
    thread.start()
    proc.wait()
    thread.join()
    progress.close()

    if proc.returncode != 0:
        print(f"FFmpeg failed for dish {dish_id}")
    else:
        print(f"Dish {dish_id} saved to {output_path}")

# === Step 4: Run all crops in parallel ===
os.makedirs(output_dir, exist_ok=True)
with ThreadPoolExecutor(max_workers=min(num_dishes, 6)) as executor:
    futures = [executor.submit(crop_dish, dish_id, roi) for dish_id, roi in dish_rois]

Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!
Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!
Select a ROI and then press SPACE or ENTER button!
Cancel the selection process by pressing c button!


KeyboardInterrupt: 



Cropping Dish 2:  55%|█████▌    | 360310/649553 [1:10:16<56:24, 85.45frame/s]


FFmpeg failed for dish 2






Cropping Dish 3:  55%|█████▌    | 360092/649553 [1:10:16<56:29, 85.40frame/s]


FFmpeg failed for dish 1
FFmpeg failed for dish 3
